In [38]:
import os
import glob
import pandas as pd
import re

In [39]:
def parse_filename(filename):

    base_name = os.path.splitext(filename)[0]  #Remove .csv
    
    label = 'nothing'
    value_r = 0.0
    value_c = 0.0

    def parse_r(val_str):
        val_str = val_str.lower().replace('ohm', '')
        if 'k' in val_str:
            return float(val_str.replace('k', '')) * 1000.0
        return float(val_str)
    
    def parse_c(val_str):
        val_str = val_str.lower().replace('nf', '')
        return float(val_str)
    
    if base_name.lower() == 'nothing':
        label = 'nothing'

    elif base_name.startswith('Pure_C_'):
        label = 'Capacitor'
        value_c = parse_c(base_name.split('Pure_C_')[1])

    elif base_name.startswith('Pure_R_'):
        label = 'Resistor'
        value_r = parse_r(base_name.split('Pure_R_')[1])

    elif base_name.startswith('Series_'):
        label = 'Series_RC'
        match = re.search(r'R_([0-9kK\.]+)_C_([0-9nNfF\.]+)', base_name)
        if match:
            value_r = parse_r(match.group(1))
            value_c = parse_c(match.group(2))
    
    elif base_name.startswith('Parallel_'):
        label = 'Parallel_RC'
        match = re.search(r'R_([0-9kK\.]+)_C_([0-9nNfF\.]+)', base_name)
        if match:
            value_r = parse_r(match.group(1))
            value_c = parse_c(match.group(2))
            
    return label, value_r, value_c

def process_data(filepath, label, value_r, value_c, output_folder='data/process_data'):
    df = pd.read_csv(filepath, names=['timestamp', 'data'], header=0)
    df = df.dropna(subset=['data'])

    df_clean = df[df['data'].str.match(r'^\d')].copy()

    split_cols = df_clean['data'].str.split(',', expand=True)

    df_clean['Index'] = split_cols[0].astype(int)
    df_clean['Magnitude'] = split_cols[1].astype(float)
    df_clean['Phase'] = split_cols[2].astype(float)
    df_clean['AccMagnitude'] = split_cols[3].astype(float)
    df_clean['AccPhase'] = split_cols[4].astype(float)

    df_clean = df_clean.drop(columns=['data', 'timestamp'])
    df_clean['Label'] = label
    df_clean['Value_R'] = value_r
    df_clean['Value_C'] = value_c

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    base_name = os.path.splitext(os.path.basename(filepath))[0]
    new_name = f"{base_name}_processed.csv"
    output_path = os.path.join(output_folder, new_name)

    df_clean.to_csv(output_path, index=False)

In [40]:
raw_data_folder = 'data/raw_data'

for filepath in glob.glob(os.path.join(raw_data_folder, '*.csv')):
    filename = os.path.basename(filepath)
    label, value_r, value_c = parse_filename(filename)
    
    print(f"Label: [{label}] | R: {value_r} Ohm | C: {value_c} nF | file: {filename}")
    
    process_data(filepath, label, value_r, value_c)

Label: [nothing] | R: 0.0 Ohm | C: 0.0 nF | file: nothing.csv
Label: [Parallel_RC] | R: 1300.0 Ohm | C: 100.0 nF | file: Parallel_R_1.3k_C_100nF.csv
Label: [Parallel_RC] | R: 1300.0 Ohm | C: 10.0 nF | file: Parallel_R_1.3k_C_10nF.csv
Label: [Parallel_RC] | R: 1300.0 Ohm | C: 22.0 nF | file: Parallel_R_1.3k_C_22nF.csv
Label: [Parallel_RC] | R: 1300.0 Ohm | C: 33.0 nF | file: Parallel_R_1.3k_C_33nF.csv
Label: [Parallel_RC] | R: 1300.0 Ohm | C: 47.0 nF | file: Parallel_R_1.3k_C_47nF.csv
Label: [Parallel_RC] | R: 1300.0 Ohm | C: 68.0 nF | file: Parallel_R_1.3k_C_68nF.csv
Label: [Parallel_RC] | R: 2400.0 Ohm | C: 100.0 nF | file: Parallel_R_2.4k_C_100nF.csv
Label: [Parallel_RC] | R: 2400.0 Ohm | C: 10.0 nF | file: Parallel_R_2.4k_C_10nF.csv
Label: [Parallel_RC] | R: 2400.0 Ohm | C: 22.0 nF | file: Parallel_R_2.4k_C_22nF.csv
Label: [Parallel_RC] | R: 2400.0 Ohm | C: 33.0 nF | file: Parallel_R_2.4k_C_33nF.csv
Label: [Parallel_RC] | R: 2400.0 Ohm | C: 47.0 nF | file: Parallel_R_2.4k_C_47nF.csv

In [41]:
def flattening_merge_data(processed_folder='data/process_data', output_file='data/final_data.csv'):
    all_files = glob.glob(os.path.join(processed_folder, "*_processed.csv"))
    all_samples = []
    global_id = 0
    
    for file in all_files:
        df = pd.read_csv(file)
        if df.empty: continue
            
        df['Sample_ID'] = (df['Index'] == 0).cumsum() + global_id
        global_id = df['Sample_ID'].max()
        
        df_pivot = df.pivot_table(
            index=['Sample_ID', 'Label', 'Value_R', 'Value_C'], 
            columns='Index', 
            values=['Magnitude', 'Phase']
        )
        
        df_pivot.columns = [f"{c[0]}_{c[1]}" for c in df_pivot.columns]
        df_pivot = df_pivot.reset_index()
        
        all_samples.append(df_pivot)
        
    final_df = pd.concat(all_samples, ignore_index=True)
    final_df.to_csv(output_file, index=False)
    return final_df

In [42]:
flattened_df = flattening_merge_data()
display(flattened_df.head())

,Sample_ID,Label,Value_R,Value_C,Magnitude_0,Magnitude_1,Magnitude_2,Magnitude_3,Magnitude_4,Magnitude_5,...,Phase_5,Phase_6,Phase_7,Phase_8,Phase_9,Phase_10,Phase_11,Phase_12,Phase_13,Phase_14
0,1,nothing,0.0,0.0,139435.593,170791.421,176258.484,135388.687,215221.218,196589.843,...,175.297,-163.614,-171.446,-163.435,-158.626,-168.840,-157.760,-161.634,-160.782,-152.673
1,2,nothing,0.0,0.0,1171894.875,163993.812,1542996.750,441958.310,333872.593,700793.187,...,-26.154,18.430,78.518,51.747,9.166,43.648,39.437,20.472,19.448,24.981
2,3,nothing,0.0,0.0,360081.312,183087.625,680353.562,378817.000,374421.812,535006.375,...,-26.310,-110.969,93.626,70.853,-12.300,81.700,14.597,9.510,16.307,-18.706
3,4,nothing,0.0,0.0,331647.343,191307.578,834786.500,354133.187,404361.750,487509.406,...,-23.742,-101.964,79.179,78.230,-16.494,87.193,23.300,4.200,-3.430,-15.823
4,5,nothing,0.0,0.0,207994.593,159652.218,1089567.250,305106.218,486771.343,875787.562,...,-16.320,138.530,-8.710,37.952,175.100,-29.863,-40.171,-44.318,-37.201,78.420


In [43]:
df = pd.read_csv('data/final_data.csv')
display(df.head())

,Sample_ID,Label,Value_R,Value_C,Magnitude_0,Magnitude_1,Magnitude_2,Magnitude_3,Magnitude_4,Magnitude_5,...,Phase_5,Phase_6,Phase_7,Phase_8,Phase_9,Phase_10,Phase_11,Phase_12,Phase_13,Phase_14
0,1,nothing,0.0,0.0,139435.593,170791.421,176258.484,135388.687,215221.218,196589.843,...,175.297,-163.614,-171.446,-163.435,-158.626,-168.840,-157.760,-161.634,-160.782,-152.673
1,2,nothing,0.0,0.0,1171894.875,163993.812,1542996.750,441958.310,333872.593,700793.187,...,-26.154,18.430,78.518,51.747,9.166,43.648,39.437,20.472,19.448,24.981
2,3,nothing,0.0,0.0,360081.312,183087.625,680353.562,378817.000,374421.812,535006.375,...,-26.310,-110.969,93.626,70.853,-12.300,81.700,14.597,9.510,16.307,-18.706
3,4,nothing,0.0,0.0,331647.343,191307.578,834786.500,354133.187,404361.750,487509.406,...,-23.742,-101.964,79.179,78.230,-16.494,87.193,23.300,4.200,-3.430,-15.823
4,5,nothing,0.0,0.0,207994.593,159652.218,1089567.250,305106.218,486771.343,875787.562,...,-16.320,138.530,-8.710,37.952,175.100,-29.863,-40.171,-44.318,-37.201,78.420
